# 03 — Experiments for the paper  ·  run on **T4 GPU**

All experiments reported in the paper. Everything runs in Colab; no Raspberry Pi is needed.

| # | Experiment | Training? | Time (T4) |
|---|---|---|---|
| E1 | **Local data ablation** — full data vs no Nigerian images vs Nigerian ×1 vs Nigerian-only | yes (4 runs) | ~1.5 h each (Nigeria-only ~15 min) |
| E2 | **Leave-one-dataset-out** — drop each source, test on every source | yes (4 runs) | ~1.5 h each |
| E3 | **Per-source accuracy** of the main YOLO26n / YOLO11n models | no | ~5 min |
| E4 | **Patch-aware scoring** of Nigerian weeds (patch labels vs individual-weed predictions) | no | ~2 min |
| E5 | **Bootstrap 95 % confidence intervals** — paired YOLO26n vs YOLO11n, and paired differences between ablation runs (full, Nigerian ×1, Nigerian-only, no Nigeria) | no | ~20 min |
| E6 | **Simplified synthetic targeting experiment** — weed threshold × maize safety margin on still test images, and k-of-5 confirmation on jittered copies (not a replay of the robot's tracker) | no | ~10 min |
| E7 | *(optional)* NCNN FP16 export accuracy | no | ~10 min |

**Google Drive folder** — `MyDrive/maize_weed/` (the same folder notebooks 01 and 02 use):
```
maize_weed/
├── maizeweed_dataset.zip              (from notebook 01)
└── runs/
    ├── yolo26n_v1/weights/best.pt     (from notebook 02, MODEL = 'yolo26n.pt')
    └── yolo11n_v1/weights/best.pt     (from notebook 02, MODEL = 'yolo11n.pt')
```
`experiments/` is created automatically.

**How to use**
1. Run **Section 1** every session (mount Drive, unzip dataset, settings).
2. Training runs (E1/E2): list the runs in `RUN_NOW` (default: all eight) and run Section 2. Each finished run writes `experiments/results/<name>.json` and is **skipped next time**; an interrupted run **resumes** from its last epoch. Do 1–2 runs per session.
3. Sections 3–7 need no training and can run any time (they use the models from notebook 02).
4. **Section 8** gathers everything into the paper figures/tables.

Everything is saved in **`MyDrive/maize_weed/experiments/`** — `results/` (JSON per run), `figures/` (PNG + PDF), tables (CSV + LaTeX) and `CAPTIONS.md`.

## 1. Setup  *(run every session)*

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics==8.4.202          # pinned: same version as the earlier evaluation runs

In [ ]:
import os, glob, json, random, shutil, time
# ---------------- settings ----------------
DRIVE_DIR   = '/content/drive/MyDrive/maize_weed'            # same folder as notebooks 01 and 02
DATASET_ZIP = f'{DRIVE_DIR}/maizeweed_dataset.zip'
OUT         = '/content/maizeweed'
BASE_MODEL  = 'yolo26n.pt'                                     # model used for the ablation runs
MAIN_MODELS = {                                                # trained in notebook 02 (YOLO11n is optional)
    'YOLO26n': f'{DRIVE_DIR}/runs/yolo26n_v1/weights/best.pt',
    'YOLO11n': f'{DRIVE_DIR}/runs/yolo11n_v1/weights/best.pt',
}
THRESHOLDS = {'YOLO26n': (0.278, 0.25), 'YOLO11n': (0.20, 0.31)}   # (maize, weed) chosen on Nigerian validation in notebook 02
SAFETY_MARGIN = 0.15                                           # maize protection zone used by the robot
ABL_EPOCHS, ABL_PATIENCE, IMGSZ, BATCH, SEED = 50, 15, 640, 32, 0   # same for every ablation run (fair comparison)
EXPERIMENTS = {   # name: what is in the training set
    'full':              'all sources, Nigerian images x3 (same data as the main model, 50 epochs)',
    'no_nigeria':        'all sources except Nigeria',
    'nigeria_x1':        'all sources, Nigerian images x1 (no oversampling)',
    'nigeria_only':      'Nigerian images only (x1)',
    'minus_cornweed':    'all except CornWeed',
    'minus_cropandweed': 'all except CropAndWeed',
    'minus_acre':        'all except ACRE',
    'minus_rf_cornweed': 'all except Roboflow corn-weed',
}
RUN_NOW = list(EXPERIMENTS)   # training runs to do (finished ones are skipped, interrupted ones resume)
# ------------------------------------------
EXP_DIR = f'{DRIVE_DIR}/experiments'; RES_DIR = f'{EXP_DIR}/results'; FIG_DIR = f'{EXP_DIR}/figures'
for d in (RES_DIR, FIG_DIR): os.makedirs(d, exist_ok=True)

if not os.path.exists(f'{OUT}/images/test'):
    if not os.path.exists(DATASET_ZIP):
        found = glob.glob('/content/drive/MyDrive/**/maizeweed_dataset*.zip', recursive=True)
        raise FileNotFoundError(f'Dataset zip not found at {DATASET_ZIP}. Found: {found}')
    !unzip -q "{DATASET_ZIP}" -d /content

def src_of(p): return os.path.basename(p).rsplit('_', 1)[0]
def imgs(split): return sorted(glob.glob(f'{OUT}/images/{split}/*.jpg'))
SPLIT = {s: imgs(s) for s in ('train', 'val', 'test')}
SOURCES = [s for s in ('nigeria', 'cornweed', 'cropandweed', 'acre', 'rf_cornweed', 'weedmaize')
           if any(src_of(p) == s for p in SPLIT['train'] + SPLIT['test'])]
SRC_NAME = {'nigeria': 'Nigeria', 'cornweed': 'CornWeed', 'cropandweed': 'CropAndWeed', 'acre': 'ACRE',
            'rf_cornweed': 'Roboflow', 'weedmaize': 'WeedMaize', 'all': 'All test'}
NAMES = 'names:\n  0: maize\n  1: weed\n'

def write_list(name, files):
    p = f'{OUT}/{name}.txt'; open(p, 'w').write('\n'.join(files)); return p
def write_yaml(name, train, val, test=None):
    p = f'{OUT}/{name}.yaml'
    open(p, 'w').write(f'path: {OUT}\ntrain: {train}\nval: {val}\n' + (f'test: {test}\n' if test else '') + NAMES); return p

# test subsets: all + one per source (for per-source and cross-dataset scores)
TEST_SETS = {'all': SPLIT['test']}
for s in SOURCES:
    f = [p for p in SPLIT['test'] if src_of(p) == s]
    if f: TEST_SETS[s] = f
TEST_YAML = {k: write_yaml(f'test_{k}', 'images/train', write_list(f'test_{k}', v), write_list(f'test_{k}', v)) for k, v in TEST_SETS.items()}
NG_VAL = [p for p in SPLIT['val'] if src_of(p) == 'nigeria']
print({s: len(v) for s, v in SPLIT.items()}, '| sources:', SOURCES)
print('test images per subset:', {k: len(v) for k, v in TEST_SETS.items()})

In [ ]:
# ---------- Figure style (used for every report figure) ----------
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
INK, INK2, GRID, SURFACE = '#0b0b0b', '#52514e', '#e4e3df', '#fcfcfb'
CLS_COLOR = {'maize': '#008300', 'weed': '#4a3aa7'}           # validated colour-blind-safe pair
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7', '#e34948']  # fixed order
BOX_COLOR = {0: (0, 200, 83), 1: (255, 64, 214)}             # bright overlays on photos: maize green, weed magenta
SEQ_CMAP = LinearSegmentedColormap.from_list('seq_blue', ['#fcfcfb', '#cde2fb', '#86b6ef', '#2a78d6', '#184f95', '#0d366b'])
mpl.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300, 'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE,
    'savefig.facecolor': SURFACE, 'font.size': 10, 'axes.titlesize': 11, 'axes.titleweight': 'bold',
    'axes.titlecolor': INK, 'axes.labelcolor': INK2, 'axes.edgecolor': GRID, 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6,
    'axes.axisbelow': True, 'xtick.color': INK2, 'ytick.color': INK2, 'legend.frameon': False,
    'lines.linewidth': 2, 'font.family': 'DejaVu Sans'})

def savefig(fig, name, show=True):
    os.makedirs(FIG_DIR, exist_ok=True)
    for ext in ('png', 'pdf'):
        fig.savefig(f'{FIG_DIR}/{name}.{ext}', bbox_inches='tight')
    print(f'saved {FIG_DIR}/{name}.png/.pdf')
    if show: plt.show()
    plt.close(fig)

def label_bars(ax, bars, fmt='{:,.0f}', inside=False):
    for b in bars:
        v = b.get_height() if b.get_width() < b.get_height() * 50 else b.get_width()
        v = b.get_height()
        if not np.isfinite(v) or v < 0: continue          # label real zeros too (e.g. weed AP 0.00)
        ax.annotate(fmt.format(v), (b.get_x() + b.get_width() / 2, v), xytext=(0, 3),
                    textcoords='offset points', ha='center', va='bottom', fontsize=8, color=INK2)

def save_table(df, name):
    os.makedirs(FIG_DIR, exist_ok=True)
    df.to_csv(f'{FIG_DIR}/{name}.csv', index=False)
    try:
        with open(f'{FIG_DIR}/{name}.tex', 'w') as f: f.write(df.to_latex(index=False, float_format='%.3f'))
    except Exception: pass
    print(f'saved {FIG_DIR}/{name}.csv/.tex')

from PIL import ImageDraw
def draw_boxes(im, boxes, width=None, colors=None):
    """boxes: list of (cls, x0, y0, x1, y1) in pixels. Returns a copy with boxes drawn."""
    im = im.copy(); d = ImageDraw.Draw(im); width = width or max(2, round(max(im.size) / 250))
    for c, x0, y0, x1, y1 in boxes:
        d.rectangle([x0, y0, x1, y1], outline=(colors or BOX_COLOR)[int(c)], width=width)
    return im

def read_label(img_path):
    im = Image.open(img_path); W, H = im.size; out = []
    lp = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    if os.path.exists(lp):
        for line in open(lp):
            c, cx, cy, w, h = map(float, line.split())
            out.append((int(c), (cx - w/2) * W, (cy - h/2) * H, (cx + w/2) * W, (cy + h/2) * H))
    return im, out

from matplotlib.patches import Patch
BOX_LEGEND = [Patch(facecolor='none', edgecolor=tuple(v/255 for v in BOX_COLOR[0]), linewidth=2, label='maize'),
              Patch(facecolor='none', edgecolor=tuple(v/255 for v in BOX_COLOR[1]), linewidth=2, label='weed')]

import numpy as np, pandas as pd
from collections import Counter, defaultdict
from PIL import Image
from ultralytics import YOLO
CLASS_NAMES = ['maize', 'weed']
captions = {}
cap_path = f'{FIG_DIR}/CAPTIONS.md'
if os.path.exists(cap_path):                                  # keep captions from earlier sessions
    import re
    for k, v in re.findall(r'\*\*(.+?)\*\* — (.+?)\n\n', open(cap_path).read(), flags=re.S): captions[k] = v
def caption(name, text):
    captions[name] = text
    with open(cap_path, 'w') as f:
        for k, v in sorted(captions.items()): f.write(f'**{k}** — {v}\n\n')
def save_json(name, obj): json.dump(obj, open(f'{RES_DIR}/{name}.json', 'w'), indent=1)
def load_json(name):
    p = f'{RES_DIR}/{name}.json'; return json.load(open(p)) if os.path.exists(p) else None

def evaluate(weights, subsets=None):
    # mAP per test subset (Ultralytics val). Returns {subset: {map50, map50_95, maize_ap50, weed_ap50, images}}
    m, out = YOLO(weights), {}
    for k in (subsets or TEST_SETS):
        r = m.val(data=TEST_YAML[k], split='test', imgsz=IMGSZ, batch=BATCH, plots=False, verbose=False,
                  project=f'{EXP_DIR}/val_tmp', name='v', exist_ok=True)
        ap50 = {CLASS_NAMES[int(c)]: float(a) for c, a in zip(r.box.ap_class_index, r.box.ap50)}
        out[k] = {'map50': float(r.box.map50), 'map50_95': float(r.box.map), 'maize_ap50': ap50.get('maize'),
                  'weed_ap50': ap50.get('weed'), 'images': len(TEST_SETS[k])}
    return out
print('helpers ready')

## 2. Training ablations (E1, E2)  *(GPU)*
Every run uses **the same settings** (`ABL_EPOCHS` epochs, same seed, validation set and augmentation), so differences come from the training data only. The `full` run is the reference for all of them.

In [ ]:
def train_files(exp):
    tr = SPLIT['train']
    base = [p for p in tr if not src_of(p).startswith('nigeriarep')]          # every image once
    if exp == 'full':         return tr
    if exp == 'no_nigeria':   return [p for p in base if src_of(p) != 'nigeria']
    if exp == 'nigeria_x1':   return base
    if exp == 'nigeria_only': return [p for p in base if src_of(p) == 'nigeria']
    if exp.startswith('minus_'):
        s = exp[len('minus_'):]
        return [p for p in tr if src_of(p) != s]
    raise ValueError(exp)

import torch
def run_finished(run_dir):
    # Ultralytics writes epoch = -1 into last.pt only when a run has really ended (all epochs done or early stopping)
    p = f'{run_dir}/weights/last.pt'
    return os.path.exists(p) and torch.load(p, map_location='cpu', weights_only=False).get('epoch', 0) == -1

for exp in RUN_NOW:
    if load_json(exp):
        print(f'[{exp}] already finished - skipped'); continue
    files = train_files(exp)
    if not files: print(f'[{exp}] no training images - skipped'); continue
    data = write_yaml(f'train_{exp}', write_list(f'train_{exp}', files), 'images/val')
    run_dir = f'{EXP_DIR}/runs/{exp}'
    print(f'\n===== {exp}: {EXPERIMENTS[exp]} - {len(files)} training images =====')
    t0 = time.time()
    if run_finished(run_dir):
        print('training already finished - evaluating only')
    elif os.path.exists(f'{run_dir}/weights/last.pt'):
        print('resuming from the last saved epoch')
        YOLO(f'{run_dir}/weights/last.pt').train(resume=True)                    # any error stops here (nothing is saved)
    else:
        YOLO(BASE_MODEL).train(data=data, epochs=ABL_EPOCHS, patience=ABL_PATIENCE, imgsz=IMGSZ, batch=BATCH, seed=SEED,
                               flipud=0.5, degrees=10, project=f'{EXP_DIR}/runs', name=exp, exist_ok=True)
    if not run_finished(run_dir):
        raise RuntimeError(f'[{exp}] training has not finished - no result saved. Run this cell again to resume it.')
    res = pd.read_csv(f'{run_dir}/results.csv')
    result = {'experiment': exp, 'description': EXPERIMENTS[exp], 'train_images': len(files),
              'train_sources': dict(Counter(src_of(p) for p in files)), 'epochs_run': len(res),
              'batches_per_epoch': -(-len(files) // BATCH),
              'model': BASE_MODEL, 'hours': round((time.time() - t0) / 3600, 2),
              'test': evaluate(f'{run_dir}/weights/best.pt')}
    save_json(exp, result)
    print(pd.DataFrame(result['test']).T.round(3))

## 3. Per-source accuracy of the main models (E3)

In [ ]:
for name, w in MAIN_MODELS.items():
    if not os.path.exists(w): print(f'{name}: {w} not found - skipped'); continue
    if load_json(f'main_{name}'): print(f'{name}: done before'); continue
    save_json(f'main_{name}', {'experiment': f'main_{name}', 'weights': w, 'test': evaluate(w)})
    print(name, 'done')
rows = []
for name in MAIN_MODELS:
    r = load_json(f'main_{name}')
    if not r: continue
    for k, v in r['test'].items():
        rows.append({'model': name, 'test subset': SRC_NAME.get(k, k), 'images': v['images'], 'maize AP50': v['maize_ap50'],
                     'weed AP50': v['weed_ap50'], 'mAP50': v['map50'], 'mAP50-95': v['map50_95']})
if rows:
    t = pd.DataFrame(rows).round(3); save_table(t, 'tableE3_per_source_accuracy'); display(t)
    caption('tableE3_per_source_accuracy', 'Test accuracy of the main models (100 epochs, notebook 02) on all test images and on the test images of each source dataset.')

## Shared: predictions for E4–E6
Runs each main model — and the finished ablation runs `full`, `nigeria_x1`, `nigeria_only`, `no_nigeria` (for E5) — once on all test images (low confidence, so thresholds can be changed afterwards) and caches the result.

In [ ]:
import pickle
def iou_xyxy(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1])); inter = ix * iy
    return inter / ((a[2]-a[0]) * (a[3]-a[1]) + (b[2]-b[0]) * (b[3]-b[1]) - inter + 1e-9)
def center_in(p, g): cx, cy = (p[0] + p[2]) / 2, (p[1] + p[3]) / 2; return g[0] <= cx <= g[2] and g[1] <= cy <= g[3]

ABL_EVAL = ['full', 'nigeria_x1', 'nigeria_only', 'no_nigeria']      # ablation checkpoints compared in E5
WEIGHTS = {n: w for n, w in MAIN_MODELS.items() if os.path.exists(w)}
for e in ABL_EVAL:
    w = f'{EXP_DIR}/runs/{e}/weights/best.pt'
    if load_json(e) and os.path.exists(w): WEIGHTS[f'abl_{e}'] = w
PRED = {}
def get_preds(name, conf=0.001):          # same settings as Ultralytics val: conf 0.001, NMS IoU 0.7, max 300
    cache = f'{RES_DIR}/preds_{name}.pkl'
    if os.path.exists(cache): return pickle.load(open(cache, 'rb'))
    m, out, files = YOLO(WEIGHTS[name]), {}, SPLIT['test']
    for k in range(0, len(files), 32):
        for fp, r in zip(files[k:k+32], m.predict(files[k:k+32], imgsz=IMGSZ, conf=conf, iou=0.7, max_det=300, verbose=False)):
            out[fp] = [(int(c), float(s), *map(float, b)) for c, s, b in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist(), r.boxes.xyxy.tolist())]
    pickle.dump(out, open(cache, 'wb')); return out
GT = {fp: list(dict.fromkeys(read_label(fp)[1])) for fp in SPLIT['test']}   # (cls, x0, y0, x1, y1) px; exact duplicates removed like Ultralytics
for name in WEIGHTS:
    PRED[name] = get_preds(name); print(name, 'predictions:', sum(len(v) for v in PRED[name].values()))
MODELS = [n for n in PRED if n in MAIN_MODELS]       # main models (have robot thresholds): E4, E5, E6
ABL = [n for n in PRED if n.startswith('abl_')]      # ablation checkpoints: E5 accuracy only
print('main models:', MODELS, '| ablation runs:', ABL)

## 4. Patch-aware scoring of Nigerian weeds (E4)
Nigerian weeds are labelled as **patches** (one box around a group of weeds); the model often predicts **individual weeds**. Standard scoring (IoU ≥ 0.5) can then count such predictions as errors. Three views (patch-aware and area are **supplementary, relaxed** measures):
- **standard** — IoU ≥ 0.5 between prediction and label
- **patch-aware** — a prediction counts if its **centre lies inside** a labelled weed patch; a patch counts as found if it contains at least one predicted weed centre. A higher score under this rule does **not** prove that each prediction is a correctly detected individual weed.
- **area covered** — share of labelled weed-patch area covered by predicted weed boxes

In [ ]:
def weed_scores(name, files, thr):
    st = Counter(); covered = total = 0
    for fp in files:
        g = [b[1:] for b in GT[fp] if b[0] == 1]
        p = [x[2:] for x in PRED[name][fp] if x[0] == 1 and x[1] >= thr]
        used = set()
        for pi, pb in enumerate(p):                       # standard greedy IoU matching
            best = max(((iou_xyxy(pb, gb), gi) for gi, gb in enumerate(g) if gi not in used), default=(0, -1))
            if best[0] >= 0.5: used.add(best[1]); st['tp'] += 1
        st['pred'] += len(p); st['gt'] += len(g)
        st['p_in'] += sum(any(center_in(pb, gb) for gb in g) for pb in p)
        st['g_hit'] += sum(any(center_in(pb, gb) for pb in p) for gb in g)
        if g:
            W, H = Image.open(fp).size; mg = np.zeros((H, W), bool); mp = np.zeros((H, W), bool)
            for b in g: mg[int(b[1]):int(np.ceil(b[3])), int(b[0]):int(np.ceil(b[2]))] = True
            for b in p: mp[int(b[1]):int(np.ceil(b[3])), int(b[0]):int(np.ceil(b[2]))] = True
            covered += (mg & mp).sum(); total += mg.sum()
    P = lambda a, b: a / b if b else float('nan')
    return {'standard precision': P(st['tp'], st['pred']), 'standard recall': P(st['tp'], st['gt']),
            'patch-aware precision': P(st['p_in'], st['pred']), 'patch-aware recall': P(st['g_hit'], st['gt']),
            'weed area covered': P(covered, total), 'weed labels': st['gt'], 'weed predictions': st['pred']}

rows = []
for name in MODELS:
    thr = THRESHOLDS[name][1]
    for sub in [s for s in ('nigeria', 'all') if s in TEST_SETS]:
        rows.append({'model': name, 'test subset': SRC_NAME[sub], 'weed threshold': thr, **weed_scores(name, TEST_SETS[sub], thr)})
if rows:
    tE4 = pd.DataFrame(rows).round(3); save_table(tE4, 'tableE4_patch_aware_weed_scoring'); display(tE4); save_json('E4', rows)
    caption('tableE4_patch_aware_weed_scoring', 'Weed precision and recall under standard IoU≥0.5 matching versus patch-aware matching (prediction centre inside a labelled weed box), and the share of labelled weed area covered by predicted weed boxes, at each model\'s robot weed threshold.')
    d = tE4[tE4['test subset'] == 'Nigeria']
    if len(d):
        keys = ['standard precision', 'patch-aware precision', 'standard recall', 'patch-aware recall', 'weed area covered']
        fig, ax = plt.subplots(figsize=(9, 4.2)); x = np.arange(len(keys)); w = 0.8 / len(d)
        for i, (_, r) in enumerate(d.iterrows()):
            label_bars(ax, ax.bar(x + (i - (len(d) - 1) / 2) * w, [r[k] for k in keys], w, color=SERIES[i], label=r['model']), fmt='{:.2f}')
        ax.set_xticks(x, [k.replace(' ', '\n', 1) for k in keys]); ax.set_ylim(0, 1.08); ax.set_ylabel('score')
        ax.set_title(f'Nigerian weeds: standard vs patch-aware scoring ({len(TEST_SETS["nigeria"])} test images)'); ax.legend()
        savefig(fig, 'figE4_patch_aware_scoring')
        caption('figE4_patch_aware_scoring', 'Weed detection on the Nigerian test images scored with standard IoU≥0.5 matching and with relaxed patch-aware matching (prediction centre inside a labelled weed patch). Patch-aware scores are a supplementary measure: a higher value shows that predicted weed centres fall inside labelled patches, not that the predictions are correctly detected individual weeds.')

## 5. Bootstrap confidence intervals (E5)
Resamples the test images 1,000 times (with replacement) and recomputes the metrics, giving **95 % confidence intervals** and **paired** differences (both models scored on the same resampled images): YOLO26n − YOLO11n, and between the ablation runs full / Nigerian ×1 / Nigerian-only / no Nigeria.
- E5 runs Ultralytics val once per model and test subset and keeps the **validator's own per-image records** (true-positive flags at IoU 0.5, confidences, classes, label counts). AP is computed with **Ultralytics' own `compute_ap`** — no re-prediction, no re-matching.
- Before resampling, the unresampled AP is compared with that same val run (**tableE5c**); E5 **stops** if any difference exceeds 0.001.
- Intervals cover **test-image sampling only**, not run-to-run (seed) variation of training.
- `maize/weed detection recall %` = labels matched one-to-one (IoU ≥ 0.5) by a detection at the robot threshold — a detection measure, not robot targeting.

In [ ]:
if not PRED: raise RuntimeError('No models found - train the main models with notebook 02 and finish the ablation runs.')
# E5 resamples the Ultralytics validator's OWN per-image records (true-positive flags, confidences, predicted classes,
# label classes) and computes AP with Ultralytics' own compute_ap. Nothing is re-predicted or re-matched.
from ultralytics.models.yolo.detect import DetectionValidator
from ultralytics.utils.metrics import compute_ap

class RecordingValidator(DetectionValidator):
    last = None
    def get_stats(self):                                            # called once, before Ultralytics clears its stats
        st = self.metrics.stats
        names = [os.path.basename(f) for f in self.dataloader.dataset.im_files]   # dataset order = record order
        RecordingValidator.last = [
            {'name': n, 'tp': np.asarray(t, bool).reshape(-1, self.niou)[:, 0].astype(float),   # IoU 0.5 column
             'conf': np.asarray(c, float).reshape(-1), 'pred_cls': np.asarray(p).reshape(-1).astype(int),
             'target_cls': np.asarray(g).reshape(-1).astype(int)}
            for n, t, c, p, g in zip(names, st['tp'], st['conf'], st['pred_cls'], st['target_cls'])]
        return super().get_stats()

def val_records(name, sub):
    cache = f'{RES_DIR}/valrecords_{name}_{sub}.pkl'
    if os.path.exists(cache): return pickle.load(open(cache, 'rb'))
    RecordingValidator.last = None
    r = YOLO(WEIGHTS[name]).val(validator=RecordingValidator, data=TEST_YAML[sub], split='test', imgsz=IMGSZ, batch=BATCH,
                                plots=False, verbose=False, project='/content/val_e5', name=f'{name}_{sub}', exist_ok=True)
    recs = RecordingValidator.last
    want = sorted(os.path.basename(p) for p in TEST_SETS[sub])
    if recs is None or sorted(x['name'] for x in recs) != want:
        raise RuntimeError(f'{name}/{sub}: validator records missing or do not match the test images - stop and report this.')
    # keep the VALIDATOR's order (Ultralytics sorts the concatenated records, so ties between equal confidences are
    # ordered exactly as in Ultralytics); file names are used only to align images across models for paired resampling
    ap50 = {CLASS_NAMES[int(c)]: float(a) for c, a in zip(r.box.ap_class_index, r.box.ap50)}
    out = {'records': recs, 'ref': {'map50': float(r.box.map50), 'maize_ap50': ap50.get('maize'), 'weed_ap50': ap50.get('weed')}}
    pickle.dump(out, open(cache, 'wb')); return out

def flatten(recs, sub):
    pos = {os.path.basename(p): i for i, p in enumerate(TEST_SETS[sub])}   # image index = position in the sorted test list
    F = {'conf': np.concatenate([x['conf'] for x in recs]), 'tp': np.concatenate([x['tp'] for x in recs]),   # validator order
         'pcls': np.concatenate([x['pred_cls'] for x in recs]),
         'img': np.concatenate([np.full(len(x['conf']), pos[x['name']]) for x in recs]).astype(int),
         'ngt': np.zeros((len(pos), 2))}
    for x in recs: F['ngt'][pos[x['name']]] = [np.sum(x['target_cls'] == c) for c in (0, 1)]
    F['order'] = np.argsort(-F['conf'])                             # same sort as Ultralytics ap_per_class
    return F

def ap50_w(F, c, w):
    # AP@0.5 for class c, each image counted w[image] times; at w = 1 this is exactly Ultralytics' calculation
    n_l = (F['ngt'][:, c] * w).sum()
    if n_l == 0: return None                                        # class absent -> not part of mAP (as Ultralytics)
    idx = F['order'][F['pcls'][F['order']] == c]
    wi = w[F['img'][idx]]; keep = wi > 0; idx, wi = idx[keep], wi[keep]
    if len(idx) == 0: return 0.0
    tp = F['tp'][idx]; tpc = np.cumsum(tp * wi); fpc = np.cumsum((1 - tp) * wi)
    return float(compute_ap(tpc / (n_l + 1e-16), tpc / (tpc + fpc))[0])

def recall_at(F, c, thr, w):
    # share of labels of class c matched (IoU >= 0.5, one-to-one, as in Ultralytics) by a detection with conf >= thr
    n_l = (F['ngt'][:, c] * w).sum(); m = (F['pcls'] == c) & (F['conf'] >= thr)
    return 100 * (F['tp'][m] * w[F['img'][m]]).sum() / n_l if n_l else None

def metrics(name, sub, w):
    F = FL[name][sub]; m, wd = ap50_w(F, 0, w), ap50_w(F, 1, w)
    out = {'maize AP50': m, 'weed AP50': wd, 'mAP50': float(np.mean([v for v in (m, wd) if v is not None]))}
    if name in THRESHOLDS:                                          # main models: detection recall at the robot thresholds
        mt, wt = THRESHOLDS[name]
        out['maize detection recall %'] = recall_at(F, 0, mt, w); out['weed detection recall %'] = recall_at(F, 1, wt, w)
    return out

SUBS = [s for s in ('all', 'nigeria') if s in TEST_SETS]
EVAL_MODELS = MODELS + ABL                 # (not NAMES - that holds the class names used by Setup)
VR = {name: {sub: val_records(name, sub) for sub in SUBS} for name in EVAL_MODELS}
FL = {name: {sub: flatten(VR[name][sub]['records'], sub) for sub in SUBS} for name in EVAL_MODELS}
LABEL = lambda n: n[len('abl_'):] if n.startswith('abl_') else n

# ---- 1. check BEFORE resampling: unresampled AP must equal the same Ultralytics val run ----
check_rows = []
for sub in SUBS:
    for name in EVAL_MODELS:
        pt, ref = metrics(name, sub, np.ones(len(TEST_SETS[sub]))), VR[name][sub]['ref']
        for k, rk in (('maize AP50', 'maize_ap50'), ('weed AP50', 'weed_ap50'), ('mAP50', 'map50')):
            if ref.get(rk) is not None and pt[k] is not None:
                check_rows.append({'test subset': SRC_NAME[sub], 'model': LABEL(name), 'metric': k, 'this notebook': pt[k],
                                   'Ultralytics': ref[rk], 'difference': pt[k] - ref[rk]})
tC = pd.DataFrame(check_rows)
save_table(tC.round(6), 'tableE5c_ap_check_vs_ultralytics'); display(tC.round(6))
caption('tableE5c_ap_check_vs_ultralytics', 'Unresampled AP@0.5 computed in this notebook from the Ultralytics validator\'s own per-image records (true-positive flags, confidences, classes) with Ultralytics\' compute_ap, versus the metrics reported by that same validation run. Must agree to within 0.001 before resampling.')
TOL = 0.001
worst = tC.difference.abs().max() if len(tC) else float('nan')
print(f'largest |difference| from Ultralytics: {worst:.6f} (tolerance {TOL})')
if not len(tC) or not worst <= TOL:
    raise RuntimeError('E5 STOPPED: the unresampled AP does not reproduce Ultralytics - send tableE5c; no intervals were computed.')

# ---- 2. paired bootstrap: same resampled images for every model ----
PAIRS = [('YOLO26n', 'YOLO11n'), ('abl_full', 'abl_nigeria_x1'), ('abl_full', 'abl_nigeria_only'),
         ('abl_nigeria_x1', 'abl_nigeria_only'), ('abl_full', 'abl_no_nigeria')]
B_ITERS, rng = 1000, np.random.default_rng(0)
rows, diff_rows, pair_check = [], [], []
for sub in SUBS:
    n = len(TEST_SETS[sub])
    point = {name: metrics(name, sub, np.ones(n)) for name in EVAL_MODELS}
    boots = {name: [] for name in EVAL_MODELS}
    for _ in range(B_ITERS):
        w = np.bincount(rng.integers(0, n, n), minlength=n).astype(float)
        for name in EVAL_MODELS: boots[name].append(metrics(name, sub, w))
    for name in EVAL_MODELS:
        for k, v in point[name].items():
            if v is None: continue
            arr = np.array([b[k] for b in boots[name] if b[k] is not None], float)
            rows.append({'test subset': SRC_NAME[sub], 'model': LABEL(name), 'metric': k, 'value': v,
                         'CI low': np.percentile(arr, 2.5), 'CI high': np.percentile(arr, 97.5)})
    for a, b in PAIRS:
        if a not in point or b not in point: continue
        ra, rb = VR[a][sub]['ref'], VR[b][sub]['ref']
        for k in point[a]:
            if k not in point[b] or point[a][k] is None or point[b][k] is None: continue
            d = np.array([x[k] - y[k] for x, y in zip(boots[a], boots[b]) if x[k] is not None and y[k] is not None])
            lo, hi = np.percentile(d, [2.5, 97.5]); dn = point[a][k] - point[b][k]
            rk = {'maize AP50': 'maize_ap50', 'weed AP50': 'weed_ap50', 'mAP50': 'map50'}.get(k)
            if rk and ra.get(rk) is not None and rb.get(rk) is not None:
                du = ra[rk] - rb[rk]; gap = dn - du
                flag = 'sign differs' if np.sign(dn) != np.sign(du) and abs(du) > TOL else ('gap > tolerance' if abs(gap) > max(0.25 * abs(du), TOL) else 'ok')
                pair_check.append({'test subset': SRC_NAME[sub], 'comparison': f'{LABEL(a)} - {LABEL(b)}', 'metric': k,
                                   'difference (this notebook)': dn, 'difference (Ultralytics)': du, 'gap': gap, 'flag': flag})
                verified = flag == 'ok'
            else:
                verified = None                                     # detection-recall rows: no Ultralytics counterpart
            diff_rows.append({'test subset': SRC_NAME[sub], 'comparison': f'{LABEL(a)} - {LABEL(b)}', 'metric': k,
                              'difference': dn, 'CI low': lo, 'CI high': hi, 'CI excludes 0': not (lo <= 0 <= hi),
                              'verified vs Ultralytics': verified})

tP = pd.DataFrame(pair_check)
if len(tP):
    save_table(tP.round(6), 'tableE5d_difference_check_vs_ultralytics'); display(tP.round(6))
    if (tP.flag != 'ok').any(): print('WARNING: these AP contrasts are NOT verified:'); display(tP[tP.flag != 'ok'])
    caption('tableE5d_difference_check_vs_ultralytics', f'For each paired AP contrast: difference computed in this notebook versus the same difference from the Ultralytics validation runs. Flags (computed before rounding): "sign differs" (when the Ultralytics difference exceeds {TOL}), or "gap > tolerance" when |gap| > max(25 % of the Ultralytics difference, {TOL} absolute); only "ok" contrasts are verified.')
else:
    print('WARNING: no AP contrast could be checked - all paired AP comparisons are UNVERIFIED.')
tE5 = pd.DataFrame(rows).round(4); save_table(tE5, 'tableE5_bootstrap_ci'); display(tE5)
caption('tableE5_bootstrap_ci', f'Point estimates and 95 % bootstrap confidence intervals ({B_ITERS} image-level resamples) from the Ultralytics validator\'s own per-image records; AP computed with Ultralytics\' compute_ap (checked against Ultralytics in tableE5c). Intervals reflect test-image sampling only, not training-run (seed) variation. "Detection recall at threshold" = share of labels of that class matched one-to-one (IoU≥0.5) by a detection at the robot threshold; a detection measure, not a robot targeting outcome.')
tD = pd.DataFrame(diff_rows)
if len(tD):
    tD = tD.round(4); save_table(tD, 'tableE5b_paired_difference'); display(tD)
    caption('tableE5b_paired_difference', 'Paired bootstrap differences (same resampled test images for both models) with 95 % confidence intervals: YOLO26n − YOLO11n, and between ablation runs (full = Nigerian ×3). "CI excludes 0" = distinguishable from 0 on this test set; training-run (seed) variation is not included. AP rows: report only if "verified vs Ultralytics" = True (tableE5d). Detection-recall rows have no Ultralytics counterpart (blank).')
    def ci_panel(ax, d, xlabel):
        d = d.reset_index(drop=True)
        for i, r in d.iterrows():
            col = SERIES[0] if r['test subset'] == 'All test' else SERIES[1]
            ax.plot([r['CI low'], r['CI high']], [i, i], color=col, linewidth=3, solid_capstyle='round')
            ax.plot(r['difference'], i, 'o', color=col, markersize=7, markeredgecolor=SURFACE)
        ax.axvline(0, color=INK2, linewidth=1); ax.invert_yaxis()
        ax.set_yticks(range(len(d)), [f"{r.comparison} · {r.metric}\n{r['test subset']}" for _, r in d.iterrows()], fontsize=8)
        ax.grid(axis='x'); ax.grid(axis='y', visible=False); ax.set_xlabel(xlabel)
    acc = ['maize AP50', 'weed AP50', 'mAP50']
    shown = tD[(tD['verified vs Ultralytics'] == True) | tD['verified vs Ultralytics'].isna()]   # figures: verified AP rows only
    dm = shown[shown.comparison == 'YOLO26n - YOLO11n']
    if len(dm):
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={'width_ratios': [3, 2]})
        ci_panel(axes[0], dm[dm.metric.isin(acc)], 'YOLO26n − YOLO11n')
        ci_panel(axes[1], dm[~dm.metric.isin(acc)], 'YOLO26n − YOLO11n (percentage points)')
        axes[0].set_title('Accuracy difference (95 % CI)'); axes[1].set_title('Detection recall at robot threshold (95 % CI)')
        plt.tight_layout(); savefig(fig, 'figE5_paired_bootstrap')
        caption('figE5_paired_bootstrap', 'Paired bootstrap differences between YOLO26n and YOLO11n (dot = point estimate, bar = 95 % CI; blue = all test images, orange = Nigerian test images). Only AP contrasts verified against Ultralytics are shown. Right panel: detection recall at each model\'s robot threshold, not robot targeting outcomes.')
    da = shown[(shown.comparison != 'YOLO26n - YOLO11n') & shown.metric.isin(acc)]
    if len(da):
        fig, ax = plt.subplots(figsize=(9, 0.32 * len(da) + 1.6))
        ci_panel(ax, da, 'difference in AP@0.5'); ax.set_title('Ablation runs: paired differences (95 % CI)')
        plt.tight_layout(); savefig(fig, 'figE5b_ablation_paired_bootstrap')
        caption('figE5b_ablation_paired_bootstrap', f'Paired bootstrap differences in AP@0.5 between ablation runs ({BASE_MODEL}, {ABL_EPOCHS} epochs; full = Nigerian ×3); only contrasts verified against Ultralytics are shown. Blue = all test images, orange = Nigerian test images. Single training run per condition: intervals cover test-image sampling only.')

## 6. Simplified synthetic targeting experiment (E6)
A **simplified** offline experiment on still test images — not a replay of the robot software. It uses static maize protection zones on single images; the robot additionally uses consecutive-frame tracking and extra filters, which are not reproduced here.
- **E6a** — sweep the **weed threshold** for several **maize safety margins**: share of weeds hit vs share of maize plants hit (a "hit" = the target centre, where the laser aims, lies inside the labelled plant box).
- **E6b** — **k-of-5 confirmation on synthetic frames**: each image is turned into 5 altered copies (small shift, brightness change, noise); a target counts only if seen in ≥ k copies. A synthetic stability test — not the robot's tracker.

In [ ]:
if not MODELS: raise RuntimeError('No main model found (see MAIN_MODELS) - train the main models with notebook 02.')
def targets(dets, mt, wt, margin, W, H):
    maize = [d[2:] for d in dets if d[0] == 0 and d[1] >= mt]
    zones = [[max(0, b[0] - margin * (b[2]-b[0])), max(0, b[1] - margin * (b[3]-b[1])),
              min(W, b[2] + margin * (b[2]-b[0])), min(H, b[3] + margin * (b[3]-b[1]))] for b in maize]
    out = []
    for d in dets:
        if d[0] != 1 or d[1] < wt: continue
        b = d[2:]
        if any(not (b[2] < z[0] or z[2] < b[0] or b[3] < z[1] or z[3] < b[1]) for z in zones): continue
        out.append(b)
    return out

def outcome(files, tgt_of):
    s = Counter()
    for fp in files:
        t = tgt_of(fp); g = GT[fp]
        mz = [b[1:] for b in g if b[0] == 0]; wd = [b[1:] for b in g if b[0] == 1]
        s['maize'] += len(mz); s['weeds'] += len(wd); s['targets'] += len(t)
        s['maize_hit'] += sum(any(center_in(tb, m) for tb in t) for m in mz)
        s['weed_hit'] += sum(any(center_in(tb, w) for tb in t) for w in wd)
        s['false'] += sum(not any(center_in(tb, w) for w in wd) for tb in t)
    return {'weeds hit %': 100 * s['weed_hit'] / max(s['weeds'], 1), 'maize plants hit %': 100 * s['maize_hit'] / max(s['maize'], 1),
            'false targets per image': s['false'] / max(len(files), 1), 'targets': s['targets']}

SIZE = {fp: Image.open(fp).size for fp in SPLIT['test']}
MARGINS = [0.0, 0.15, 0.3, 0.5]
WTS = sorted(set(np.round(np.arange(0.10, 0.91, 0.05), 2).tolist()) | {THRESHOLDS[n][1] for n in MODELS})   # grid + exact robot thresholds
rows = []
for name in MODELS:
    mt, wt0 = THRESHOLDS[name]
    for sub in [s for s in ('all', 'nigeria') if s in TEST_SETS]:
        for mg in MARGINS:
            for wt in WTS:
                r = outcome(TEST_SETS[sub], lambda fp: targets(PRED[name][fp], mt, wt, mg, *SIZE[fp]))
                rows.append({'model': name, 'test subset': SRC_NAME[sub], 'margin': mg, 'weed threshold': wt, **r})
tE6 = pd.DataFrame(rows).round(3); save_table(tE6, 'tableE6a_robot_sweep'); save_json('E6a', rows)
caption('tableE6a_robot_sweep', 'Simplified synthetic targeting experiment on still test images (static protection zones; no tracking or additional robot filters): weeds hit, maize plants hit (target centre inside the plant box) and false targets per image, for each weed threshold and maize safety margin, at each model\'s maize threshold.')
main = MODELS[0]
subs = [s for s in ('All test', 'Nigeria') if s in set(tE6['test subset'])]
fig, axes = plt.subplots(1, len(subs), figsize=(6 * len(subs), 4.4), squeeze=False)
for ax, sub in zip(axes[0], subs):
    for i, mg in enumerate(MARGINS):
        d = tE6[(tE6.model == main) & (tE6['test subset'] == sub) & (tE6.margin == mg)]
        ax.plot(d['maize plants hit %'], d['weeds hit %'], '-o', markersize=3, color=SERIES[i], label=f'margin {mg:.2f}')
    op = tE6[(tE6.model == main) & (tE6['test subset'] == sub) & (tE6.margin == SAFETY_MARGIN)]
    op = op[np.isclose(op['weed threshold'], THRESHOLDS[main][1])]                   # exact robot threshold
    ax.plot(op['maize plants hit %'], op['weeds hit %'], '*', markersize=15, color=INK, label='robot setting')
    ax.set_xlabel('maize plants hit (%)'); ax.set_ylabel('weeds hit (%)'); ax.set_title(f'{main} - {sub}'); ax.grid(axis='x')
axes[0][0].legend(fontsize=8)
plt.tight_layout(); savefig(fig, 'figE6a_robot_tradeoff')
caption('figE6a_robot_tradeoff', f'Simplified synthetic targeting trade-off for {main} on still test images: weeds hit versus maize plants hit as the weed threshold varies (0.10 to 0.90 in steps of 0.05, plus each model\'s exact robot threshold), for different maize safety margins. Star = thresholds and margin used on the robot (weed threshold {THRESHOLDS[main][1]}, margin {SAFETY_MARGIN}).')
ops = []
for (mname, sub), d in tE6[tE6.margin == SAFETY_MARGIN].groupby(['model', 'test subset']):
    ops.append(d[np.isclose(d['weed threshold'], THRESHOLDS[mname][1])])
print('At the robot settings:'); display(pd.concat(ops))

In [ ]:
if not MODELS: raise RuntimeError('No main model found (see MAIN_MODELS) - train the main models with notebook 02.')
# E6b - multi-frame confirmation with 5 jittered frames per image
N_FRAMES, JIT = 5, np.random.default_rng(1)
def jitter(img, k):
    if k == 0: return img, 0, 0
    a = np.asarray(img).astype(np.float32); H, W = a.shape[:2]
    dx, dy = int(JIT.uniform(-0.02, 0.02) * W), int(JIT.uniform(-0.02, 0.02) * H)
    a = np.roll(np.roll(a, dy, 0), dx, 1) * JIT.uniform(0.85, 1.15) + JIT.normal(0, 4, a.shape)
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8)), dx, dy

def frame_targets(name, files):
    cache = f'{RES_DIR}/frames_{name}.pkl'
    if os.path.exists(cache): return pickle.load(open(cache, 'rb'))
    m, mt, wt = YOLO(MAIN_MODELS[name]), *THRESHOLDS[name]; out = {}
    for fp in files:
        img = Image.open(fp).convert('RGB'); W, H = img.size; fr, sh = [], []
        for k in range(N_FRAMES):
            im, dx, dy = jitter(img, k); fr.append(im); sh.append((dx, dy))
        res = m.predict(fr, imgsz=IMGSZ, conf=min(mt, wt), verbose=False)
        out[fp] = []
        for r, (dx, dy) in zip(res, sh):
            dets = [(int(c), float(s), b[0] - dx, b[1] - dy, b[2] - dx, b[3] - dy)
                    for c, s, b in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist(), r.boxes.xyxy.tolist())]
            out[fp].append(targets(dets, mt, wt, SAFETY_MARGIN, W, H))
    pickle.dump(out, open(cache, 'wb')); return out

def confirmed(frames, k, W):
    # targets from frame 0 that are seen (centre within 4 % of width) in at least k of the frames
    out = []
    for t in frames[0]:
        c = ((t[0] + t[2]) / 2, (t[1] + t[3]) / 2); seen = 0
        for f in frames:
            seen += any(np.hypot((u[0] + u[2]) / 2 - c[0], (u[1] + u[3]) / 2 - c[1]) < 0.04 * W for u in f)
        if seen >= k: out.append(t)
    return out

rows = []
for name in MODELS:
    FR = frame_targets(name, SPLIT['test'])
    for sub in [s for s in ('all', 'nigeria') if s in TEST_SETS]:
        for k in range(1, N_FRAMES + 1):
            rows.append({'model': name, 'test subset': SRC_NAME[sub], 'frames required': k,
                         **outcome(TEST_SETS[sub], lambda fp: confirmed(FR[fp], k, SIZE[fp][0]))})
tE6b = pd.DataFrame(rows).round(3); save_table(tE6b, 'tableE6b_multiframe_confirmation'); display(tE6b); save_json('E6b', rows)
caption('tableE6b_multiframe_confirmation', f'Effect of requiring a target in k of {N_FRAMES} jittered frames (±2 % shift, ±15 % brightness, noise) at the robot thresholds and safety margin {SAFETY_MARGIN}. A simplified synthetic stability test on altered still images; it does not reproduce the robot\'s consecutive-frame tracker or additional filters.')
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, key in zip(axes, ['weeds hit %', 'maize plants hit %', 'false targets per image']):
    for i, (lab, d) in enumerate(tE6b[tE6b.model == MODELS[0]].groupby('test subset')):
        ax.plot(d['frames required'], d[key], '-o', color=SERIES[i], label=lab)
    ax.set_xlabel(f'frames required (of {N_FRAMES})'); ax.set_title(key); ax.set_xticks(range(1, N_FRAMES + 1))
axes[0].legend(fontsize=8); plt.tight_layout(); savefig(fig, 'figE6b_multiframe_confirmation')
caption('figE6b_multiframe_confirmation', f'{MODELS[0]}: weeds hit, maize plants hit and false targets per image versus the number of synthetically altered copies of each test image in which a target must appear (simplified stability test, not the robot tracker).')

## 7. *(Optional)* FP16 export accuracy (E7)
Accuracy only — speed cannot be measured without the Raspberry Pi.

In [ ]:
DO_E7 = False   # set True to run
if DO_E7 and 'YOLO26n' in MAIN_MODELS and os.path.exists(MAIN_MODELS['YOLO26n']):
    rows = []
    for tag, kw in (('NCNN FP32', {}), ('NCNN FP16', {'half': True})):   # NCNN INT8 is not supported by Ultralytics export
        try:
            p = YOLO(MAIN_MODELS['YOLO26n']).export(format='ncnn', imgsz=416, **kw)
            dst = f'{EXP_DIR}/export_{tag.split()[1].lower()}_ncnn_model'; shutil.rmtree(dst, ignore_errors=True); shutil.move(p, dst)
            mb = sum(os.path.getsize(f) for f in glob.glob(f'{dst}/*')) / 1e6
            ev = {}
            for k in [s for s in ('all', 'nigeria') if s in TEST_SETS]:
                r = YOLO(dst, task='detect').val(data=TEST_YAML[k], split='test', imgsz=416, batch=1, plots=False, verbose=False,
                                                 project=f'{EXP_DIR}/val_tmp', name='v', exist_ok=True)
                ev[k] = float(r.box.map50)
            rows.append({'format': tag, 'size MB': round(mb, 2), 'mAP50 all test': ev.get('all'), 'mAP50 Nigeria': ev.get('nigeria')})
        except Exception as e:
            rows.append({'format': tag, 'note': f'not supported: {str(e)[:80]}'})
    tE7 = pd.DataFrame(rows); save_table(tE7, 'tableE7_quantisation'); display(tE7)
    caption('tableE7_quantisation', 'YOLO26n exported to NCNN at 416 px in FP32 and FP16 (INT8 is not available for NCNN in Ultralytics): file size and test mAP@0.5 (accuracy only).')

## 8. Summary — ablation figures and tables (E1, E2)
Uses every finished run in `experiments/results/`.

In [ ]:
R = {e: load_json(e) for e in EXPERIMENTS}; R = {k: v for k, v in R.items() if v}
print('finished runs:', list(R) or 'none yet')
if R:
    rows = []
    for e, r in R.items():
        csv = f'{EXP_DIR}/runs/{e}/results.csv'
        ep = len(pd.read_csv(csv)) if os.path.exists(csv) else r['epochs_run']
        bpe = -(-r['train_images'] // BATCH)                        # batches per epoch
        for k, v in r['test'].items():
            rows.append({'experiment': e, 'training data': r['description'], 'train images': r['train_images'], 'epochs': ep,
                         'batches/epoch': bpe, 'batch iterations': bpe * ep,
                         'test subset': SRC_NAME.get(k, k), 'maize AP50': v['maize_ap50'], 'weed AP50': v['weed_ap50'], 'mAP50': v['map50'], 'mAP50-95': v['map50_95']})
    tAll = pd.DataFrame(rows).round(3); save_table(tAll, 'tableE1E2_ablation_all_subsets')
    caption('tableE1E2_ablation_all_subsets', f'All ablation runs ({BASE_MODEL}, up to {ABL_EPOCHS} epochs, identical settings): test accuracy on all images and on each source\'s test images. Equal epochs mean unequal training: "batch iterations" (batches per epoch × epochs) differ with training-set size (e.g. Nigerian-only).')

    # E1 - local data ablation
    e1 = [e for e in ('full', 'nigeria_x1', 'no_nigeria', 'nigeria_only') if e in R]
    if e1:
        t1 = tAll[tAll.experiment.isin(e1) & tAll['test subset'].isin(['All test', 'Nigeria'])]
        save_table(t1, 'tableE1_local_data_ablation'); display(t1)
        caption('tableE1_local_data_ablation', 'Effect of Nigerian training data: full data (Nigerian ×3), Nigerian ×1, no Nigerian images, Nigerian images only. Test on all images and on Nigerian test images.')
        LAB = {'full': 'full\n(Nigeria ×3)', 'nigeria_x1': 'Nigeria ×1', 'no_nigeria': 'no Nigeria', 'nigeria_only': 'Nigeria only'}
        fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
        for ax, key in zip(axes, ['mAP50', 'maize AP50', 'weed AP50']):
            x = np.arange(len(e1)); w = 0.38
            for j, sub in enumerate(['All test', 'Nigeria']):
                vals = [t1[(t1.experiment == e) & (t1['test subset'] == sub)][key].squeeze() if len(t1[(t1.experiment == e) & (t1['test subset'] == sub)]) else 0 for e in e1]
                label_bars(ax, ax.bar(x + (j - 0.5) * w, np.nan_to_num(np.array(vals, float)), w, color=SERIES[j], label=f'{sub} images'), fmt='{:.2f}')
            ax.set_xticks(x, [LAB[e] for e in e1], fontsize=9); ax.set_title(key); ax.set_ylim(0, 1.08)
        axes[0].legend(fontsize=8); axes[0].set_ylabel('AP@0.5')
        plt.tight_layout(); savefig(fig, 'figE1_local_data_ablation')
        caption('figE1_local_data_ablation', f'Local data ablation ({BASE_MODEL}, {ABL_EPOCHS} epochs): accuracy on all test images and Nigerian test images for different amounts of Nigerian training data.')

    # E2 - leave-one-dataset-out heatmap
    e2 = [e for e in ['full', 'no_nigeria'] + [f'minus_{s}' for s in SOURCES if s != 'nigeria'] if e in R]
    if len(e2) > 1:
        cols = ['all'] + [s for s in SOURCES if s in TEST_SETS]
        for key, fname in (('map50', 'figE2_leave_one_out_map50'), ('weed_ap50', 'figE2b_leave_one_out_weed')):
            cell = lambda e, c: R[e]['test'].get(c, {}).get(key)
            M = np.array([[np.nan if cell(e, c) is None else cell(e, c) for c in cols] for e in e2], float)   # keep real 0.0 scores
            fig, ax = plt.subplots(figsize=(1.3 * len(cols) + 3, 0.6 * len(e2) + 1.8))
            im = ax.imshow(M, cmap=SEQ_CMAP, vmin=0, vmax=1, aspect='auto')
            for i in range(len(e2)):
                for j in range(len(cols)):
                    if np.isnan(M[i, j]): continue
                    ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=9, color='white' if M[i, j] > 0.6 else INK)
                held = e2[i].replace('minus_', '') if e2[i].startswith('minus_') else ('nigeria' if e2[i] == 'no_nigeria' else None)
                if held in cols:
                    j = cols.index(held); ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor=SERIES[1], linewidth=2.5))
            ax.set_xticks(range(len(cols)), [SRC_NAME.get(c, c) for c in cols]); ax.set_yticks(range(len(e2)), [e.replace('minus_', 'without ').replace('no_nigeria', 'without nigeria').replace('_', ' ') for e in e2])
            ax.set_xlabel('test images from'); ax.grid(False)
            ax.set_title(('mAP@0.5' if key == 'map50' else 'weed AP@0.5') + ' when one source is left out of training')
            fig.colorbar(im, ax=ax, fraction=0.03); savefig(fig, fname)
            caption(fname, f'Leave-one-dataset-out ({BASE_MODEL}, {ABL_EPOCHS} epochs): {"mAP@0.5" if key == "map50" else "weed AP@0.5"} on each source\'s test images. Orange box = the source that was left out of training (cross-dataset generalisation).')

In [ ]:
# zip all experiment figures + tables for download
shutil.make_archive(f'{EXP_DIR}/experiments_figures_tables', 'zip', FIG_DIR)
print('saved', f'{EXP_DIR}/experiments_figures_tables.zip')